# Installation and Import

- macOS: Run 'brew install poppler'
- Ubuntu/Debian: Run 'apt-get install poppler-utils'
- Windows: Install from https://github.com/oschwartz10612/poppler-windows/releases/ and add to PATH

In [ ]:
# !pip install typhoon-ocr==0.4.1
# !pip install python-dotenv
# !pip install lxml

In [ ]:
from dotenv import load_dotenv
import os
import pandas as pd
from typhoon_ocr import ocr_document
from io import StringIO
import re

load_dotenv()

# OCR

In [ ]:
API_KEY = os.getenv("TYPHOON_OCR_API_KEY")

In [ ]:
data_json = ocr_document(
    api_key=API_KEY,
    pdf_or_image_path="data/district/advance/5-17 เขต 1 จ.บุรีรัมย์.pdf",
    page_num=3,             # Page 3
    target_image_dim=2400
)

In [ ]:
print(data_json)

In [ ]:
good_card = re.search(r'บัตรดี\s+([\d,]+)\s*บัตร', data_json)
bad_card = re.search(r'บัตรเสีย\s+([\d,]+)\s*บัตร', data_json)
none_card = re.search(r'บัตรที่ไม่เลือกผู้สมัครผู้ใด\s+([\d,]+)\s*บัตร', data_json)

summary = {
    'บัตรดี': int(good_card.group(1).replace(',', '')) if good_card else None,
    'บัตรเสีย': int(bad_card.group(1).replace(',', '')) if bad_card else None,
    'ไม่เลือกผู้ใด': int(none_card.group(1).replace(',', '')) if none_card else None,
}

print(summary)

In [ ]:
df = pd.read_html(StringIO(data_json))[0]
df

In [ ]:
df.columns = ['หมายเลข', 'ชื่อผู้สมัคร', 'พรรค', 'คะแนน']
df = df.iloc[1:].reset_index(drop=True)
df = df[df['ชื่อผู้สมัคร'].notna() & (df['หมายเลข'] != 'รวมคะแนนทั้งสิ้น')].reset_index(drop=True)
df['คะแนน_ตัวเลข'] = df['คะแนน'].str.extract(r'(\d+)').astype(int)
df.drop(columns=['คะแนน'], inplace=True)

df